# LeetCode #1020: Number of Enclaves

https://leetcode.com/problems/number-of-enclaves/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(m^2 n^2)$ | $O(mn)$ |
| **Optimal: Border Flood-Fill DFS ★** | $O(mn)$ | $O(mn)$ |

---

## Understanding the Methods

### Brute Force
For every land cell, launch a separate DFS/BFS to determine whether it touches the border. This re-explores overlapping regions repeatedly, leading to $O(m^2 n^2)$ work.

### Optimal: Border Flood-Fill DFS ★
Start DFS from every land cell on the four borders and mark all reachable land cells as visited (they can "escape"). Count remaining unvisited land cells — those are the enclaves.

**Why this is better than Brute Force:** Single-pass flood-fill visits each cell exactly once ($O(mn)$) instead of re-running a search per cell.

**Constraints:**
* $1 \le m, n \le 500$
* $grid[i][j] \in \{0, 1\}$

## Solutions

### C#

In [ ]:
public class Solution {
    int m, n;
    int[][] grid;
    public int NumEnclaves(int[][] grid) {
        this.grid = grid;
        m = grid.Length; n = grid[0].Length;
        // Flood-fill from every border land cell to mark escapable cells
        for (int r = 0; r < m; r++) { Dfs(r, 0); Dfs(r, n - 1); }
        for (int c = 0; c < n; c++) { Dfs(0, c); Dfs(m - 1, c); }
        // Count remaining land cells — they cannot reach the border
        int count = 0;
        for (int r = 0; r < m; r++)
            for (int c = 0; c < n; c++)
                if (grid[r][c] == 1) count++;
        return count;
    }
    void Dfs(int r, int c) {
        if (r < 0 || r >= m || c < 0 || c >= n || grid[r][c] != 1) return;
        // Mark as visited so it is excluded from the enclave count
        grid[r][c] = 0;
        Dfs(r - 1, c); Dfs(r + 1, c); Dfs(r, c - 1); Dfs(r, c + 1);
    }
}

### Python

In [ ]:
class Solution:
    def num_enclaves(self, grid: list[list[int]]) -> int:
        m, n = len(grid), len(grid[0])
        def dfs(r, c):
            if r < 0 or r >= m or c < 0 or c >= n or grid[r][c] != 1:
                return
            # Mark as visited so it is excluded from the enclave count
            grid[r][c] = 0
            dfs(r-1, c); dfs(r+1, c); dfs(r, c-1); dfs(r, c+1)
        # Flood-fill from every border land cell to mark escapable cells
        for r in range(m): dfs(r, 0); dfs(r, n-1)
        for c in range(n): dfs(0, c); dfs(m-1, c)
        return sum(grid[r][c] for r in range(m) for c in range(n))

### Go

In [ ]:
func numEnclaves(grid [][]int) int {
    m, n := len(grid), len(grid[0])
    var dfs func(r, c int)
    dfs = func(r, c int) {
        if r < 0 || r >= m || c < 0 || c >= n || grid[r][c] != 1 { return }
        // Mark as visited so it is excluded from the enclave count
        grid[r][c] = 0
        dfs(r-1, c); dfs(r+1, c); dfs(r, c-1); dfs(r, c+1)
    }
    // Flood-fill from every border land cell to mark escapable cells
    for r := 0; r < m; r++ { dfs(r, 0); dfs(r, n-1) }
    for c := 0; c < n; c++ { dfs(0, c); dfs(m-1, c) }
    count := 0
    for r := 0; r < m; r++ {
        for c := 0; c < n; c++ {
            if grid[r][c] == 1 { count++ }
        }
    }
    return count
}

### Rust

In [ ]:
impl Solution {
    pub fn num_enclaves(mut grid: Vec<Vec<i32>>) -> i32 {
        let m = grid.len();
        let n = grid[0].len();
        fn dfs(grid: &mut Vec<Vec<i32>>, r: i32, c: i32, m: i32, n: i32) {
            if r < 0 || r >= m || c < 0 || c >= n || grid[r as usize][c as usize] != 1 { return; }
            // Mark as visited so it is excluded from the enclave count
            grid[r as usize][c as usize] = 0;
            dfs(grid, r-1, c, m, n); dfs(grid, r+1, c, m, n);
            dfs(grid, r, c-1, m, n); dfs(grid, r, c+1, m, n);
        }
        let (mi, ni) = (m as i32, n as i32);
        // Flood-fill from every border land cell to mark escapable cells
        for r in 0..mi { dfs(&mut grid, r, 0, mi, ni); dfs(&mut grid, r, ni-1, mi, ni); }
        for c in 0..ni { dfs(&mut grid, 0, c, mi, ni); dfs(&mut grid, mi-1, c, mi, ni); }
        grid.iter().flatten().sum()
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `grid = [[0,0,0,0],[1,0,1,0],[0,1,1,0],[0,0,0,0]]`
Border flood-fill reaches no internal cells. The cluster of three 1s in the interior cannot touch any border, so all three count. Answer: **3**.

### 2. Slightly Complex
**Input:** `grid = [[0,1,1,0],[0,0,1,0],[0,0,1,0],[0,0,0,0]]`
The top-border cells at columns 1 and 2 are land; DFS from row-0 border reaches the entire connected component. After flood-fill, zero land cells remain. Answer: **0**.

### 3. Edge Case: Time Factor
**Input:** $500 \times 500$ grid entirely filled with 1s except for the border.
DFS visits up to $500 \times 500 = 250{,}000$ cells; each visited once, so runtime stays $O(mn)$ even for the largest input.

### 4. Edge Case: Space Factor
**Input:** $500 \times 500$ all-1s grid.
The call stack depth equals the path length of DFS traversal, which can reach $O(mn)$ in a snake-shaped path — the implicit recursion stack is the dominant space cost.

### 5. Almost-Impossible but Plausible
**Input:** Entire grid is 0 ($m = n = 500$, all water).
DFS never enters a cell (all values are 0); the final count loop returns 0 immediately. Answer: **0**.